# 🤖 Active Learning Experiment — Track A

**Цель:** сравнить стратегии **entropy** vs **random**  
**Вопрос:** сколько примеров экономит entropy при том же качестве?

| Параметр | Значение |
|---|---|
| Стартовый labeled set | N = 20 |
| Итерации AL | 5 |
| Batch size | 5 |
| Стратегии | entropy, random |
| Модель | Logistic Regression + TF-IDF |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from agents.al_agent import ActiveLearningAgent

print('✅ Imports OK')

## 1. Load Data

In [ ]:
# Пробуем загрузить датасет из пайплайна
# Ищем в нескольких возможных путях
CANDIDATES = [
    '../data/labeled/final_dataset.csv',
    '../data/labeled/reviewed.csv',
    '../data/labeled/auto_labeled.csv',
    '../data/labeled_dataset.csv',
]

df = None
TEXT_COL  = 'text'
LABEL_COL = 'label_auto'

for path in CANDIDATES:
    if os.path.exists(path):
        df = pd.read_csv(path)
        # убираем ошибки разметки
        if LABEL_COL in df.columns:
            df = df[df[LABEL_COL] != 'error'].dropna(subset=[LABEL_COL])
        if TEXT_COL not in df.columns:
            # попробуем найти текстовую колонку
            for c in ('review','sentence','comment','content'):
                if c in df.columns:
                    df = df.rename(columns={c: TEXT_COL})
                    break
        if TEXT_COL in df.columns and LABEL_COL in df.columns and len(df) > 10:
            print(f'✅ Loaded from: {path}  ({len(df)} rows)')
            break
        df = None

if df is None:
    print('Pipeline dataset not found — using 20 Newsgroups demo (3 classes, ~500 rows)')
    cats = ['sci.med', 'rec.sport.hockey', 'comp.graphics']
    news = fetch_20newsgroups(subset='all', categories=cats,
                              remove=('headers','footers','quotes'))
    df = pd.DataFrame({
        'text':  [t[:400] for t in news.data],
        'label_auto': [news.target_names[t] for t in news.target]
    }).dropna()
    # Обрезаем до 300 для скорости
    df = df.sample(300, random_state=42).reset_index(drop=True)
    TEXT_COL, LABEL_COL = 'text', 'label_auto'
    print(f'Loaded 20NG demo: {len(df)} rows')

df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
print(f'\nFinal shape: {df.shape}')
print('Label distribution:')
print(df[LABEL_COL].value_counts())

## 2. Setup & Split (адаптивный под размер датасета)

In [ ]:
n = len(df)

# Адаптивные параметры — работают и на 100, и на 1000 строках
N_TEST      = max(15, int(0.2 * n))       # 20% на тест, минимум 15
N_START     = max(10, int(0.15 * n))      # 15% стартовых меток, минимум 10
BATCH_SIZE  = max(3,  int(0.05 * n))      # 5% за итерацию, минимум 3
N_ITER      = 5
STRATEGIES  = ['entropy', 'random']

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

test_df       = df.iloc[-N_TEST:]
rest_df       = df.iloc[:-N_TEST]
labeled_start = rest_df.iloc[:N_START]
pool_df       = rest_df.iloc[N_START:]

print(f'Dataset size:      {n}')
print(f'Start labeled:     {len(labeled_start)}')
print(f'Pool:              {len(pool_df)}')
print(f'Test:              {len(test_df)}')
print(f'Batch size:        {BATCH_SIZE}')
print(f'Max new labels:    {N_ITER * BATCH_SIZE} (per strategy)')

if len(pool_df) < BATCH_SIZE:
    print('\n⚠️  Pool меньше batch_size — уменьшаем batch')
    BATCH_SIZE = max(1, len(pool_df) // N_ITER)
    print(f'   New BATCH_SIZE = {BATCH_SIZE}')

## 3. Run Active Learning — Both Strategies

In [ ]:
agent = ActiveLearningAgent()

# Обучаем общий vectorizer на всех текстах
all_text = pd.concat([rest_df, test_df])[TEXT_COL].astype(str)
agent.vectorizer.fit(all_text)
agent._fitted_vectorizer = True

histories = {}

for strategy in STRATEGIES:
    print(f'\n{"="*50}')
    print(f'Strategy: {strategy.upper()}')
    print('='*50)

    agent.model = LogisticRegression(max_iter=1000, C=1.0)

    history = agent.run_cycle(
        labeled_df   = labeled_start.copy(),
        pool_df      = pool_df.copy(),
        test_df      = test_df,
        strategy     = strategy,
        n_iterations = N_ITER,
        batch_size   = BATCH_SIZE,
        text_col     = TEXT_COL,
        label_col    = LABEL_COL,
    )
    histories[strategy] = history
    print(f'  → {len(history)} iterations recorded')

print('\n✅ AL cycles complete')

## 4. Results Table

In [ ]:
frames = []
for strategy, history in histories.items():
    if not history:
        print(f'⚠️  History for {strategy} is empty — skipping')
        continue
    df_h = pd.DataFrame(history)
    df_h['strategy'] = strategy
    frames.append(df_h)

if not frames:
    raise RuntimeError('All histories are empty — check dataset size and column names')

results_df = pd.concat(frames, ignore_index=True)
print('Columns in results_df:', list(results_df.columns))
print(f'Total rows: {len(results_df)}')
print()

# Показываем таблицу результатов
display_cols = [c for c in ['strategy','iteration','n_labeled','accuracy','f1'] if c in results_df.columns]
print(results_df[display_cols].to_string(index=False))

# Pivot (только если есть данные по обеим стратегиям)
if len(results_df['strategy'].unique()) > 1:
    pivot = results_df.pivot_table(
        index='n_labeled',
        columns='strategy',
        values=['accuracy', 'f1']
    ).round(4)
    print('\nPivot table:')
    display(pivot)
else:
    print('\n(Only one strategy ran — pivot skipped)')

## 5. Learning Curves Plot

In [ ]:
# Sample savings
if 'entropy' in histories and 'random' in histories and histories['entropy'] and histories['random']:
    savings = ActiveLearningAgent.compute_sample_savings(
        histories['entropy'], histories['random']
    )
else:
    savings = {}
print('Sample savings:', json.dumps(savings, indent=2))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Active Learning: Entropy vs Random', fontsize=14, fontweight='bold')

COLORS = {'entropy': '#667eea', 'random': '#f39c12'}
STYLES = {'entropy': '-o', 'random': '--s'}

for metric, ax, title in zip(
    ['accuracy', 'f1'],
    axes,
    ['Accuracy', 'F1-score (macro)']
):
    for strategy, history in histories.items():
        if not history:
            continue
        df_h = pd.DataFrame(history)
        if metric not in df_h.columns:
            continue
        ax.plot(
            df_h['n_labeled'], df_h[metric],
            STYLES.get(strategy, '-o'),
            color=COLORS.get(strategy, 'gray'),
            label=strategy.capitalize(), lw=2, ms=7
        )

    if metric == 'accuracy' and savings.get('target_accuracy'):
        ax.axhline(savings['target_accuracy'], color='red', ls=':', lw=1.5,
                   label=f"Target ({savings['target_accuracy']:.3f})")
        for key, strat in [('n_labeled_entropy','entropy'), ('n_labeled_random','random')]:
            nv = savings.get(key)
            if nv:
                ax.axvline(nv, color=COLORS.get(strat,'gray'), ls=':', lw=1.2, alpha=0.6)
                ax.annotate(str(nv),
                            xy=(nv, savings['target_accuracy']),
                            xytext=(nv+0.5, savings['target_accuracy']-0.03),
                            fontsize=8, color=COLORS.get(strat,'gray'))

    ax.set_xlabel('Labeled samples', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/learning_curve.png', dpi=150, bbox_inches='tight')
print('✅ Saved → ../reports/learning_curve.png')
plt.show()

## 6. Conclusion — Sample Savings

In [ ]:
print('=' * 55)
print('CONCLUSION: Active Learning Sample Savings')
print('=' * 55)

# Всегда печатаем финальные метрики
for strategy, history in histories.items():
    if not history:
        print(f'{strategy}: no history')
        continue
    last = history[-1]
    print(f"{strategy:10s}: acc={last.get('accuracy','?'):.4f}  "
          f"f1={last.get('f1','?'):.4f}  n_labeled={last.get('n_labeled','?')}")

print()
if savings.get('samples_saved') is not None:
    print(f"Target accuracy:        {savings['target_accuracy']:.4f}")
    print(f"Entropy reaches target: {savings['n_labeled_entropy']} samples")
    print(f"Random reaches target:  {savings['n_labeled_random']} samples")
    print(f"Samples SAVED:          {savings['samples_saved']} ({savings['savings_pct']}%)")
    print(f"\n→ Entropy saves {savings['savings_pct']}% of labeling effort vs random baseline.")
else:
    print('Target not reached within these iterations.')
    print('Try increasing N_ITER or BATCH_SIZE in cell 2.')

## 7. Save Report

In [ ]:
report = {
    'config': {
        'n_start':      N_START,
        'n_iterations': N_ITER,
        'batch_size':   BATCH_SIZE,
        'strategies':   STRATEGIES,
        'dataset_size': n,
        'test_size':    len(test_df),
    },
    'histories': histories,
    'savings':   savings,
}

os.makedirs('../reports', exist_ok=True)
with open('../reports/al_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('✅ reports/al_report.json')
print('✅ reports/learning_curve.png')